In [3]:
import math
import numpy as np
import torch

# ============================================================
# 4D phi^4 Langevin + DECAY EXTRACTION TOOL (STANDALONE)
# ============================================================

# ----------------------------
# PARAMETERS
# ----------------------------
L = 64
d = 4

m2 = 0.05        # bare mass^2
lam = 1.0        # quartic coupling
dt = 0.01
n_steps = 20000
burnin = 5000
sample_every = 20

# Fit / plateau windows (safe defaults for L=64)
r_fit_min, r_fit_max = 6, 20
r_plateau_min, r_plateau_max = 8, 24

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float64

# ----------------------------
# LATTICE UTILS
# ----------------------------
def laplacian(phi: torch.Tensor) -> torch.Tensor:
    out = torch.zeros_like(phi)
    for mu in range(d):
        out += torch.roll(phi,  1, dims=mu) + torch.roll(phi, -1, dims=mu) - 2.0 * phi
    return out

# ----------------------------
# SIMULATION: Langevin
# ----------------------------
torch.manual_seed(0)
phi = 0.1 * torch.randn((L,) * d, device=device, dtype=dtype)

samples = []
noise_scale = math.sqrt(2.0 * dt)

for t in range(n_steps):
    lap = laplacian(phi)
    drift = lap - m2 * phi - lam * phi**3
    phi = phi + dt * drift + noise_scale * torch.randn_like(phi)

    if t >= burnin and ((t - burnin) % sample_every == 0):
        samples.append(phi.detach().clone())

samples = torch.stack(samples, dim=0)  # (Ns, L, L, L, L)
Ns = samples.shape[0]
print(f"[device] {device}  dtype={dtype}")
print(f"[samples] Ns={Ns}  shape={tuple(samples.shape)}")

# ----------------------------
# CORRELATOR: translational average via FFT
# ----------------------------
# For each sample:
#   C(x) = <phi(0) phi(x)>_trans = IFFT( |FFT(phi)|^2 )
# Then average over samples.
C = torch.zeros((L,) * d, device=device, dtype=dtype)

for s in range(Ns):
    ph = samples[s]
    F = torch.fft.fftn(ph)
    C += torch.fft.ifftn((F.conj() * F).real).real  # |F|^2 then ifft

C /= float(Ns)

# ----------------------------
# EXTRACT POINT vs PROJECTED
# ----------------------------
C_pt = C[:, 0, 0, 0].detach().cpu().numpy().astype(np.float64)
C_proj = C.sum(dim=(1, 2, 3)).detach().cpu().numpy().astype(np.float64)

# Normalize to 1 at r=0 (doesn't affect slopes)
C_pt /= C_pt[0]
C_proj /= C_proj[0]

# ----------------------------
# DECAY EXTRACTION TOOL
# ----------------------------
EPS = 1e-300

def _clip_window(Lcorr: int, rmin: int, rmax: int) -> tuple[int, int]:
    rmin = int(max(1, rmin))
    rmax = int(min(rmax, (Lcorr // 2) - 2, Lcorr - 1))
    if rmax <= rmin:
        return rmin, rmin
    return rmin, rmax

def fit_eta_linear_log(corr: np.ndarray, rmin: int, rmax: int) -> float:
    r = np.arange(rmin, rmax + 1, dtype=np.int64)
    y = np.log(np.maximum(np.abs(corr[r]), EPS))
    A = np.vstack([np.ones_like(r, dtype=np.float64), -r.astype(np.float64)]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[1])

def fit_eta_point_corrected_4d(corr: np.ndarray, rmin: int, rmax: int) -> float:
    # log G(r) = a - eta r - (3/2) log r  -> regress (log G + 1.5 log r) on r
    r_int = np.arange(rmin, rmax + 1, dtype=np.int64)
    r = r_int.astype(np.float64)
    y = np.log(np.maximum(np.abs(corr[r_int]), EPS))
    y_corr = y + 1.5 * np.log(r)
    A = np.vstack([np.ones_like(r, dtype=np.float64), -r]).T
    coef, *_ = np.linalg.lstsq(A, y_corr, rcond=None)
    return float(coef[1])

def effective_mass(corr: np.ndarray) -> np.ndarray:
    # eta_eff(r) = log(C(r)/C(r+1))
    return np.log(np.maximum(corr[:-1], EPS) / np.maximum(corr[1:], EPS))

# Clip windows safely
rmin_fit, rmax_fit = _clip_window(L, r_fit_min, r_fit_max)

eta_proj = fit_eta_linear_log(C_proj, rmin_fit, rmax_fit)
eta_pt_raw = fit_eta_linear_log(C_pt, rmin_fit, rmax_fit)
eta_pt_corr = fit_eta_point_corrected_4d(C_pt, rmin_fit, rmax_fit)

meff = effective_mass(C_proj)
pmin = int(max(1, r_plateau_min))
pmax = int(min(r_plateau_max, (L // 2) - 3, meff.size - 1))
plateau = meff[pmin : pmax + 1] if pmax > pmin else np.array([np.nan])

# ----------------------------
# REPORT
# ----------------------------
print("\n=== INTERACTING φ⁴ DECAY EXTRACTION (4D) ===")
print(f"m2={m2}  lambda={lam}  dt={dt}  steps={n_steps}  burnin={burnin}  sample_every={sample_every}")
print(f"fit window: r=[{rmin_fit},{rmax_fit}]   plateau window: r=[{pmin},{pmax}]")
print()
print(f"eta_proj (PHYSICAL MASS PROXY)      = {eta_proj:.6f}")
print(f"eta_pt_raw (inflated point-to-point)= {eta_pt_raw:.6f}")
print(f"eta_pt_corrected (4D prefactor)     = {eta_pt_corr:.6f}")
print(f"proj effective-mass plateau mean    = {float(np.nanmean(plateau)):.6f} ± {float(np.nanstd(plateau)):.6f}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 128.12 MiB is free. Process 37790 has 14.61 GiB memory in use. Of the allocated memory 14.50 GiB is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)